# Medical Text Analysis with Anchor Explanations

This notebook demonstrates using Anchor explanations with an Ollama medical model to understand model predictions.

First, install required packages:
```bash
pip install anchor-exp spacy requests numpy
python -m spacy download en_core_web_lg
```

In [1]:
!pip install anchor-exp spacy requests numpy
!python -m spacy download en_core_web_lg


[notice] A new release of pip is available: 25.0.1 -> 25.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 25.1
[notice] To update, run: pip install --upgrade pip
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/400.7 MB ? eta -:--:--  Downloading https://github.com/explosion/spacy-models/releases/download/en_core_web_lg-3.8.0/en_core_web_lg-3.8.0-py3-none-any.whl (400.7 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 2.7 MB/s eta 0:00:0000:0100:04
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 2.7 MB/s eta 0:00:0000:01

[notice] A new release of pip is available: 25.0.1 -> 25.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 25.1
[notice] To update, run: pip install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
✔ Download and installation successful
You can now load the pa

In [2]:
import sys
import os
import numpy as np
import spacy
import requests
import json
from anchor import anchor_text
sys.path.append('..')

# Initialize spaCy
nlp = spacy.load('en_core_web_lg')



In [ ]:
# Load required JavaScript dependencies for Anchor visualization
from IPython.display import HTML, display

def load_js_dependencies():
    """Load JavaScript dependencies for Anchor visualization"""
    this_dir = os.path.dirname(anchor_text.__file__)
    bundle_path = os.path.join(this_dir, 'bundle.js')
    with open(bundle_path, encoding='utf8') as f:
        bundle = f.read()
    return HTML(f'''<script>{bundle}</script>''')

# Display JavaScript dependencies
display(load_js_dependencies())

## Define Ollama Model Call
Creating a function to interact with the Ollama API for medical data analysis.

In [ ]:
def query_ollama_model(prompt, model="medical"):
    """
    Send a query to Ollama model and get response
    """
    url = "http://localhost:11434/api/generate"
    
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False
    }
    
    try:
        response = requests.post(url, json=payload)
        return response.json()['response']
    except Exception as e:
        return f"Error: {str(e)}"

In [ ]:
# Ollama model configuration
MODEL_CONFIG = {
    "url": "http://localhost:11434/api/chat",
    "model": "llama3-med42-8b",
    "system_prompt": "You are a medical expert assistant analyzing clinical information."
}

def get_model_response(text, stream=False):
    """Get response from Ollama model"""
    try:
        payload = {
            "model": MODEL_CONFIG["model"],
            "messages": [
                {"role": "system", "content": MODEL_CONFIG["system_prompt"]},
                {"role": "user", "content": text}
            ],
            "stream": stream
        }
        
        response = requests.post(MODEL_CONFIG["url"], json=payload)
        if response.status_code == 200:
            data = response.json()
            return data.get("message", {}).get("content", "")
        return f"Error: {response.status_code}"
    except Exception as e:
        return f"Error: {str(e)}"

# Initialize medical processor
medical_processor = MedicalTermProcessor()

## Test the Model with Medical Data
Let's test the model with a sample medical dataset query.

In [ ]:
# Sample medical query
test_prompt = """
Analyze the following patient symptoms and provide potential diagnoses:
- Persistent headache
- Fatigue
- Mild fever
- Joint pain
Please format the response as JSON with likelihood percentages.
"""

# Get model response
response = query_ollama_model(test_prompt)
print("Model Response:", response)

## Process Model Output
Converting the model's response into a structured format for visualization.

In [ ]:
# Convert string response to structured data
# Note: This is a simplified example - actual processing will depend on model output
try:
    # Assuming response is in JSON format
    data = json.loads(response)
except:
    # If not JSON, create sample data for demonstration
    data = {
        "Viral Infection": 65,
        "Rheumatoid Arthritis": 45,
        "Chronic Fatigue Syndrome": 40,
        "Migraine": 75
    }

# Create DataFrame
df = pd.DataFrame(list(data.items()), columns=['Diagnosis', 'Likelihood'])

## Create Medical Data Visualization
Creating a bar chart to visualize the diagnosis likelihoods.

In [ ]:
# Set styling
plt.style.use('seaborn')
plt.figure(figsize=(10, 6))

# Create bar plot
sns.barplot(data=df, x='Likelihood', y='Diagnosis')
plt.title('Diagnosis Likelihood Analysis')
plt.xlabel('Likelihood (%)')
plt.ylabel('Potential Diagnosis')

# Add value labels
for i, v in enumerate(df['Likelihood']):
    plt.text(v, i, f' {v}%', va='center')

plt.tight_layout()
plt.show()

In [ ]:
def predict_medical(texts):
    """Prediction function for Anchor"""
    predictions = []
    
    for text in texts:
        try:
            # Get model response
            response = get_model_response(text)
            
            # Calculate medical relevance
            words = set(text.lower().split())
            medical_terms = medical_processor.medical_terms
            medical_matches = words & medical_terms
            
            # Binary classification: 1 if medical terms present, 0 if not
            prediction = 1 if len(medical_matches) > 0 else 0
            predictions.append(prediction)
            
        except Exception as e:
            print(f"Prediction error: {e}")
            predictions.append(0)
    
    return np.array(predictions)

# Initialize Anchor explainer
explainer = anchor_text.AnchorText(
    nlp=nlp,
    class_names=['not_medical', 'medical'],
    use_unk_distribution=False  # Use BERT-based distribution
)

## Generate and Visualize Explanations

Now we'll create functions to generate and visualize Anchor explanations for medical text.

In [ ]:
def explain_medical_text(text, threshold=0.95):
    """Generate Anchor explanation for medical text"""
    print(f"Analyzing: {text}")
    
    # Get model response
    response = get_model_response(text)
    print(f"\nModel response: {response}")
    
    # Generate explanation
    exp = explainer.explain_instance(
        text,
        predict_medical,
        threshold=threshold,
        verbose=False
    )
    
    return exp, response

## Test Cases

Let's test the explanation system with some medical examples:

In [ ]:
# Test with a medical example
test_prompt = "I have a severe headache and feeling dizzy with nausea"
exp, response = explain_medical_text(test_prompt)

if exp is not None:
    print('\nAnchor Rules:')
    print(' AND '.join(exp.names()))
    print(f'Precision: {exp.precision():.2f}')
    print(f'Coverage: {exp.coverage():.2f}')
    
    # Show visualization
    exp.show_in_notebook()

In [ ]:
# Test with another example
test_prompt = "Patient presents with chest pain and shortness of breath"
exp, response = explain_medical_text(test_prompt)

if exp is not None:
    print('\nAnchor Rules:')
    print(' AND '.join(exp.names()))
    print(f'Precision: {exp.precision():.2f}')
    print(f'Coverage: {exp.coverage():.2f}')
    
    # Show visualization
    exp.show_in_notebook()